# ConvNeXt-Large — From Scratch in PyTorch
**Paper:** A ConvNet for the 2020s (Liu et al., 2022)

| Config | Value |
|--------|-------|
| Depths | (3, 3, 27, 3) |
| Dims   | (192, 384, 768, 1536) |
| Params | ~198M |

In [ ]:
# Cell 1 — Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

In [ ]:
# Cell 2 — Custom LayerNorm (channels_first & channels_last)
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-6, data_format='channels_last'):
        super().__init__()
        self.weight           = nn.Parameter(torch.ones(normalized_shape))
        self.bias             = nn.Parameter(torch.zeros(normalized_shape))
        self.eps              = eps
        self.data_format      = data_format
        self.normalized_shape = (normalized_shape,)

    def forward(self, x):
        if self.data_format == 'channels_last':
            return F.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        mean = x.mean(1, keepdim=True)
        var  = (x - mean).pow(2).mean(1, keepdim=True)
        x    = (x - mean) / torch.sqrt(var + self.eps)
        return self.weight[:, None, None] * x + self.bias[:, None, None]

In [ ]:
# Cell 3 — ConvNeXt Block
class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, layer_scale_init=1e-6):
        super().__init__()
        self.dwconv  = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm    = LayerNorm(dim, eps=1e-6, data_format='channels_last')
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act     = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma   = nn.Parameter(layer_scale_init * torch.ones(dim)) if layer_scale_init > 0 else None

    def forward(self, x):
        residual = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        if self.gamma is not None:
            x = self.gamma * x
        x = x.permute(0, 3, 1, 2)
        return residual + x

In [ ]:
# Cell 4 — Full ConvNeXt Model
class ConvNeXt(nn.Module):
    def __init__(self, in_channels=3, num_classes=1000,
                 depths=(3, 3, 27, 3), dims=(192, 384, 768, 1536), layer_scale_init=1e-6):
        super().__init__()

        # Downsampling stem + 3 intermediate downsamples
        self.downsample_layers = nn.ModuleList()
        stem = nn.Sequential(
            nn.Conv2d(in_channels, dims[0], kernel_size=4, stride=4),
            LayerNorm(dims[0], eps=1e-6, data_format='channels_first'),
        )
        self.downsample_layers.append(stem)
        for i in range(3):
            ds = nn.Sequential(
                LayerNorm(dims[i], eps=1e-6, data_format='channels_first'),
                nn.Conv2d(dims[i], dims[i + 1], kernel_size=2, stride=2),
            )
            self.downsample_layers.append(ds)

        # 4 stages of ConvNeXt blocks
        self.stages = nn.ModuleList()
        for i in range(4):
            stage = nn.Sequential(
                *[ConvNeXtBlock(dims[i], layer_scale_init) for _ in range(depths[i])]
            )
            self.stages.append(stage)

        self.norm = nn.LayerNorm(dims[-1], eps=1e-6)
        self.head = nn.Linear(dims[-1], num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        for i in range(4):
            x = self.downsample_layers[i](x)
            x = self.stages[i](x)
        x = x.mean([-2, -1])
        x = self.norm(x)
        return self.head(x)

In [ ]:
# Cell 5 — Instantiate ConvNeXt-Large
NUM_CLASSES = 10
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

model = ConvNeXt(
    in_channels=3,
    num_classes=NUM_CLASSES,
    depths=(3, 3, 27, 3),
    dims=(192, 384, 768, 1536),
).to(DEVICE)
print(model)

In [ ]:
# Cell 6 — Forward Pass Test
dummy  = torch.randn(2, 3, 224, 224).to(DEVICE)
output = model(dummy)
print(f'Input  shape : {dummy.shape}')
print(f'Output shape : {output.shape}')

In [ ]:
# Cell 7 — Count Parameters
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{'='*45}")
print(f"  Model           : ConvNeXt-Large")
print(f"  Total params    : {total:,}")
print(f"  Trainable       : {trainable:,}")
print(f"  Non-trainable   : {total - trainable:,}")
print(f"{'='*45}")

In [ ]:
# Cell 8 — Layer-wise Parameter Breakdown
print(f"{'Layer':<50} {'Shape':<25} {'Params':>10}")
print('-' * 87)
for name, param in model.named_parameters():
    print(f"{name:<50} {str(list(param.shape)):<25} {param.numel():>10,}")
print('-' * 87)
print(f"{'TOTAL':<76} {total:>10,}")

---
## Training

In [ ]:
# Cell 10 — Dataset & DataLoader
DATA_DIR   = './data'
BATCH_SIZE = 8

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_transform)
val_dataset   = datasets.ImageFolder(f'{DATA_DIR}/val',   transform=val_transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

CLASS_NAMES = train_dataset.classes
print(f'Classes    : {CLASS_NAMES}')
print(f'Train size : {len(train_dataset)}')
print(f'Val size   : {len(val_dataset)}')

In [ ]:
# Cell 10 — Training Loop
EPOCHS = 20
LR     = 5e-4

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if train: optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if train: loss.backward(); optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += outputs.max(1)[1].eq(labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, 100.0 * correct / total

print(f"{'Epoch':<8} {'Tr Loss':<10} {'Tr Acc':<10} {'Val Loss':<10} {'Val Acc':<10}")
print('-' * 50)
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step()
    history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc);  history['val_acc'].append(vl_acc)
    print(f'{epoch:<8} {tr_loss:<10.4f} {tr_acc:<10.2f} {vl_loss:<10.4f} {vl_acc:<10.2f}')

torch.save(model.state_dict(), 'convnext_large_best.pth')
print('Model saved: convnext_large_best.pth')

---
## Training Curves

In [ ]:
# Cell 11 — Training Curves (Loss & Accuracy)
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ConvNeXt-Large — Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, history['train_loss'], 'b-o', linewidth=2, markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', linewidth=2, markersize=4, label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'b-o', linewidth=2, markersize=4, label='Train Acc')
axes[1].plot(epochs_range, history['val_acc'],   'r-o', linewidth=2, markersize=4, label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

---
## Inference

In [ ]:
# Cell 12 — Inference on a Single Image
from PIL import Image

def predict_image(image_path, top_k=5):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    img    = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1)

    top_probs, top_idx = torch.topk(probs, top_k, dim=1)
    top_probs = top_probs[0].cpu().tolist()
    top_idx   = top_idx[0].cpu().tolist()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title('Input Image')
    labels = [CLASS_NAMES[i] for i in top_idx]
    colors = ['#F44336' if i == 0 else '#2196F3' for i in range(top_k)]
    bars   = axes[1].barh(labels[::-1], [p*100 for p in top_probs[::-1]], color=colors[::-1])
    axes[1].set_xlabel('Confidence (%)'); axes[1].set_title(f'Top-{top_k} Predictions')
    for bar, prob in zip(bars, top_probs[::-1]):
        axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                     f'{prob*100:.1f}%', va='center')
    plt.tight_layout()
    plt.savefig('inference_result.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Predicted: {CLASS_NAMES[top_idx[0]]}  ({top_probs[0]*100:.2f}%)')

# predict_image('your_image.jpg')
print('Call: predict_image("your_image.jpg")')

---
## ROC AUC Curve

In [ ]:
# Cell 13 — Collect Validation Predictions
model.eval()
all_probs  = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        probs = F.softmax(model(images.to(DEVICE)), dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

all_probs  = np.concatenate(all_probs,  axis=0)
all_labels = np.concatenate(all_labels, axis=0)
print(f'Predictions shape : {all_probs.shape}')
print(f'Labels shape      : {all_labels.shape}')

In [ ]:
# Cell 14 — ROC AUC Curve (One-vs-Rest, per class)
y_bin   = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(10, 8))
colors  = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))

roc_auc_scores = {}
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc     = auc(fpr, tpr)
    roc_auc_scores[CLASS_NAMES[i]] = roc_auc
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'{CLASS_NAMES[i]}  (AUC = {roc_auc:.3f})')

macro_auc = roc_auc_score(y_bin, all_probs, average='macro', multi_class='ovr')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title(f'ConvNeXt-Large — ROC AUC Curve (Macro AUC = {macro_auc:.3f})', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Macro AUC : {macro_auc:.4f}')
print('Per-class AUC:')
for cls, score in roc_auc_scores.items():
    print(f'  {cls:<20} {score:.4f}')